# 03. Camada Silver: Limpeza e Padronização
Aplica os tratamentos definidos no diagnóstico de qualidade (`02_qualidade`): tipagem correta, remoção de colunas fora do escopo, tratamento de nulos e regra de negócio de venda válida. Apenas as 3 tabelas usadas pelo objetivo são levadas para a Silver.

In [0]:
%sql
CREATE OR REPLACE TABLE mvp_olist.silver.orders AS
SELECT
  order_id,
  customer_id,
  order_status,
  CAST(order_purchase_timestamp      AS TIMESTAMP) AS order_purchase_ts,
  CAST(order_approved_at             AS TIMESTAMP) AS order_approved_ts,
  CAST(order_delivered_carrier_date  AS TIMESTAMP) AS delivered_carrier_ts,
  CAST(order_delivered_customer_date AS TIMESTAMP) AS delivered_customer_ts,
  CAST(order_estimated_delivery_date AS DATE)      AS estimated_delivery_date,
  order_status NOT IN ('canceled', 'unavailable')  AS is_venda_valida
FROM mvp_olist.bronze.orders;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE mvp_olist.silver.order_items AS
SELECT
  order_id,
  CAST(order_item_id AS INT)                 AS order_item_id,
  product_id,
  seller_id,
  CAST(shipping_limit_date AS TIMESTAMP)     AS shipping_limit_ts,
  CAST(price         AS DECIMAL(10,2))       AS price,
  CAST(freight_value AS DECIMAL(10,2))       AS freight_value
FROM mvp_olist.bronze.order_items;

ALTER TABLE mvp_olist.silver.order_items ADD CONSTRAINT chk_price_positivo CHECK (price > 0);
ALTER TABLE mvp_olist.silver.order_items ADD CONSTRAINT chk_frete_nao_negativo CHECK (freight_value >= 0);

In [0]:
%sql
CREATE OR REPLACE TABLE mvp_olist.silver.products AS
SELECT
  p.product_id,
  COALESCE(p.product_category_name, 'sem_categoria')                               AS categoria_pt,
  COALESCE(t.product_category_name_english, p.product_category_name, 'sem_categoria') AS categoria_en,
  CAST(p.product_weight_g  AS INT) AS peso_g,
  CAST(p.product_length_cm AS INT) AS comprimento_cm,
  CAST(p.product_height_cm AS INT) AS altura_cm,
  CAST(p.product_width_cm  AS INT) AS largura_cm
FROM mvp_olist.bronze.products p
LEFT JOIN mvp_olist.bronze.category_translation t
  ON p.product_category_name = t.product_category_name;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT 'orders' AS tabela, (SELECT COUNT(*) FROM mvp_olist.bronze.orders) AS bronze, (SELECT COUNT(*) FROM mvp_olist.silver.orders) AS silver
UNION ALL SELECT 'order_items', (SELECT COUNT(*) FROM mvp_olist.bronze.order_items), (SELECT COUNT(*) FROM mvp_olist.silver.order_items)
UNION ALL SELECT 'products', (SELECT COUNT(*) FROM mvp_olist.bronze.products), (SELECT COUNT(*) FROM mvp_olist.silver.products)
UNION ALL SELECT 'produtos sem_categoria', NULL, (SELECT COUNT(*) FROM mvp_olist.silver.products WHERE categoria_pt = 'sem_categoria')
UNION ALL SELECT 'vendas não válidas', NULL, (SELECT COUNT(*) FROM mvp_olist.silver.orders WHERE NOT is_venda_valida);

tabela,bronze,silver
orders,99441,99441
order_items,112650,112650
products,32951,32951
produtos sem_categoria,null,610
vendas não válidas,null,1234
